# 02: Imbalanced Classification, Resampling & TreeSHAP Explainability

**Track 05: Classification & Tabular Gradient Boosting** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Solve extreme class imbalance in financial churn and fraud data. Compare class-weighted LightGBM, compute Shapley values (TreeSHAP), and generate force and summary feature impact plots.


## 1. Class Imbalance & Cost-Sensitive Learning
When minority class represents $< 10\%$, standard accuracy is misleading. We prioritize PR-AUC and Recall.

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score
import lightgbm as lgb
import shap

df = load_dataset("telecom_churn")

cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    if col != "Churn":
        df[col] = df[col].astype("category").cat.codes

if df["Churn"].dtype == object:
    df["Churn"] = (df["Churn"] == "Yes").astype(int)

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Churn Imbalance Ratio: {y_train.mean()*100:.2f}% positive class")

## 2. Class-Weighted LightGBM Classifier
Incorporate `scale_pos_weight` to penalize false negatives.

In [ ]:
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

clf = lgb.LGBMClassifier(
    n_estimators=120,
    learning_rate=0.05,
    scale_pos_weight=pos_weight,
    random_state=42,
    verbose=-1
)
clf.fit(X_train, y_train)

y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("=== LightGBM Imbalanced Classification Report ===")
print(classification_report(y_test, y_pred))
print(f"PR-AUC (Average Precision): {average_precision_score(y_test, y_prob):.4f}")
print(f"ROC-AUC Score             : {roc_auc_score(y_test, y_prob):.4f}")

## 3. Model Explainability with TreeSHAP
Compute exact Shapley values to interpret global and local feature contributions.

In [ ]:
# Compute feature importance and Shapley values
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': clf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print('=== Top 5 Most Impactful Features (LightGBM) ===')
print(feature_importance.head(5).to_string(index=False))

try:
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_test.iloc[:100])
    if isinstance(shap_values, list):
        shap_vals_matrix = shap_values[1]
    else:
        shap_vals_matrix = shap_values
    mean_abs_shap = np.abs(shap_vals_matrix).mean(axis=0)
    print('TreeSHAP Mean Absolute Attribution Computed Successfully.')
except Exception as e:
    print(f'SHAP Summary Note: {e}')